In [1]:
import sys
import pandas as pd
sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr
import metatlas2.load_tools as ldt
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa
import metatlas2.rt_align_tools as rat
import metatlas2.targeted_analysis as tga
import metatlas2.targeted_gui as tgi
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
ATLAS_TYPES = ["ISTD", "QC"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
REPORT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}_comprehensive_report.xlsx'

In [3]:
new_main_database = True
new_atlases = True
new_rt_alignment = True
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

Overwriting existing main database (overwrite_existing=True)
Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/20 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/65 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/85 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/373 [00:00<?, ?it/s]

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


Preparing compounds:   0%|          | 0/418 [00:00<?, ?it/s]

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC, ATLAS_TYPE in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC, ATLAS_TYPES):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, ATLAS_TYPE, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-948261b5838e415f9feb3ef62e19e0f6"
    TARGET_ATLAS_UID = "atl-raw-4239aecf4f51488f9b16b9329d5f8f31"


In [6]:
TARGET_ATLAS_UID

'atl-raw-ISTD-8d812bf22b8a4e42af17b0e77cab6db8'

In [7]:
QC_ATLAS_UID

'atl-raw-QC-8733960f481b4b6bb43329c2edf02f4b'

In [ ]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-58ded23f08ba48b2abce38a078b8a9b4"

In [ ]:
if new_analysis is True:
    atlas_df_ft, project_data = tga.run_targeted_analysis_workflow(PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG)
    gui = tgi.create_gui(project_data, CONFIG)
    display(gui)

In [ ]:
for inchi, inchi_data in project_data.compounds.items():
    if inchi == 'AGPKZVBTJJNPAG-WHFBIAKZSA-N':
        print(inchi_data.compound_name)
        print(inchi_data.adduct)
        print(inchi_data.mz)
        print(inchi_data.rt_peak)
        print(inchi_data.rt_min)
        print(inchi_data.rt_max)
        print(inchi_data.ms1_notes)
        print(inchi_data.ms2_notes)
        print(inchi_data.analyst_notes)
        print(inchi_data.identification_notes)

In [ ]:
#if new_analysis is True:
    #post_analysis_atlas_uid = tga.create_post_analysis_atlas_v2(project_data, CONFIG, output_dir="~/Downloads/")
    #targeted_analysis_uid = dbi.deposit_targeted_analysis_from_plot_data(atlas_df_ft, PROJECT_DB_PATH, plot_data, post_analysis_atlas_uid, PROJECT_NAME)
    #comprehensive_report = dbi.generate_comprehensive_targeted_analysis_report(PROJECT_DB_PATH, CONFIG, targeted_analysis_uid, atlas_df_ft, REPORT_PATH)

In [ ]:
dbi.validate_database(CONFIG)